# S3 — Agent Engineering: Ejercicios de código

**Módulo:** Semester 3 — Agent Engineering  
**Lecciones de referencia:** 3.2 Skills y herramientas · 3.3 Verifiers · 3.4 Supervisores · 3.5 Self-correction · 3.6 Memoria de agentes

## Objetivo

Implementar los cuatro componentes estructurales de un agente de producción:
el espacio de acción con niveles de riesgo, el loop de self-correction,
el supervisor que decide cuándo escalar, y la capa de memoria por ciclo de vida.

## Instrucciones

- Cada ejercicio tiene una celda de **starter code** (funciones con `pass`/`raise NotImplementedError`) y una celda de **tests**.
- Implementa el starter code hasta que todos los `assert` de los tests pasen sin excepción.
- Los tests **no requieren API key real** — usan mocks y valores locales.
- Orden recomendado: 1 → 2 → 3 → 4 (el 4 integra todo).


In [ ]:
# Dependencias — ejecuta esta celda primero
# pip install openai tiktoken  # solo necesario si conectas con API real

import os
import json
import time
import tempfile
import shutil
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple
from dataclasses import dataclass, field
from enum import Enum

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'sk-test')
print('Entorno listo. API key configurada:', OPENAI_API_KEY[:7] + '...')


---

## Ejercicio 1 — ToolRegistry: espacio de acción con niveles de riesgo

**Lección de referencia:** 3.2 — Skills, herramientas y límites del agente

En producción, no todas las herramientas de un agente tienen el mismo riesgo.
Leer datos de una BD es reversible y seguro. Enviar un email o ejecutar un pago no lo es.
Un `ToolRegistry` bien diseñado aplica controles distintos según el nivel de riesgo
de cada herramienta, implementando el principio de mínimo privilegio.

### Lo que debes implementar

1. `ToolRegistry.register(name, func, risk_level)` — registra una herramienta con su nivel.
2. `ToolRegistry.execute(tool_name, args, require_confirmation)` — ejecuta con las reglas:
   - `read_only`: ejecuta siempre, sin confirmación.
   - `write_reversible`: ejecuta si `require_confirmation=False` o si se llamó con confirmación explícita.
   - `irreversible`: **siempre** lanza `ConfirmationRequired` con la descripción de la acción.

### Criterios de éxito
- Las herramientas `read_only` ejecutan sin importar `require_confirmation`.
- Las herramientas `irreversible` **nunca** ejecutan directamente, siempre elevan la excepción.
- `ConfirmationRequired` incluye el nombre de la herramienta y los args en el mensaje.


In [ ]:
class ConfirmationRequired(Exception):
    """Raised when an irreversible tool is called without explicit approval."""
    pass


class RiskLevel(str, Enum):
    READ_ONLY = "read_only"
    WRITE_REVERSIBLE = "write_reversible"
    IRREVERSIBLE = "irreversible"


@dataclass
class Tool:
    name: str
    func: Callable
    risk_level: RiskLevel


class ToolRegistry:
    """
    Registro de herramientas del agente con control de riesgo.
    Implementa el principio de mínimo privilegio: cada herramienta
    declara su nivel de riesgo y el registro aplica controles distintos.
    """

    def __init__(self):
        # TODO: inicializa el diccionario interno de herramientas
        pass

    def register(self, name: str, func: Callable, risk_level: str) -> None:
        """
        Registra una herramienta en el registry.

        Args:
            name: identificador único de la herramienta
            func: callable que ejecuta la herramienta
            risk_level: uno de 'read_only', 'write_reversible', 'irreversible'
        """
        # TODO: valida que risk_level sea un RiskLevel válido
        # TODO: almacena la herramienta como Tool(name, func, RiskLevel(risk_level))
        pass

    def execute(self, tool_name: str, args: Dict[str, Any], require_confirmation: bool = False) -> Any:
        """
        Ejecuta una herramienta respetando su nivel de riesgo.

        Reglas:
        - read_only: ejecuta siempre.
        - write_reversible: ejecuta (require_confirmation no bloquea, es informativo).
        - irreversible: SIEMPRE lanza ConfirmationRequired con tool_name y args en el mensaje.

        Returns:
            El resultado de func(**args)

        Raises:
            KeyError: si tool_name no está registrado.
            ConfirmationRequired: si la herramienta es irreversible.
        """
        # TODO: obtén la herramienta del diccionario interno
        # TODO: aplica la lógica según risk_level
        pass

    def list_tools(self) -> List[Dict]:
        """Devuelve la lista de herramientas con nombre y nivel de riesgo."""
        # TODO: devuelve [{"name": ..., "risk_level": ...}, ...]
        pass


In [ ]:
# ── Tests Ejercicio 1 ────────────────────────────────────────────────────────

registry = ToolRegistry()

# Herramientas de ejemplo
def get_order_status(order_id: str) -> dict:
    return {"order_id": order_id, "status": "shipped"}

def update_delivery_date(order_id: str, new_date: str) -> bool:
    return True

def send_refund(order_id: str, amount: float) -> str:
    return f"Refund of {amount} sent for {order_id}"

registry.register("get_order_status", get_order_status, "read_only")
registry.register("update_delivery_date", update_delivery_date, "write_reversible")
registry.register("send_refund", send_refund, "irreversible")

# Test 1: read_only siempre ejecuta
result = registry.execute("get_order_status", {"order_id": "ORD-001"})
assert result == {"order_id": "ORD-001", "status": "shipped"}, "read_only debe ejecutar y devolver el resultado"

# Test 2: read_only ignora require_confirmation
result2 = registry.execute("get_order_status", {"order_id": "ORD-002"}, require_confirmation=True)
assert result2["status"] == "shipped", "read_only debe ejecutar incluso con require_confirmation=True"

# Test 3: write_reversible ejecuta sin confirmación
result3 = registry.execute("update_delivery_date", {"order_id": "ORD-003", "new_date": "2025-01-15"})
assert result3 is True, "write_reversible debe ejecutar"

# Test 4: irreversible SIEMPRE lanza ConfirmationRequired
raised = False
try:
    registry.execute("send_refund", {"order_id": "ORD-004", "amount": 99.99})
except ConfirmationRequired as e:
    raised = True
    assert "send_refund" in str(e), "ConfirmationRequired debe incluir el nombre de la herramienta"
assert raised, "irreversible debe lanzar ConfirmationRequired siempre"

# Test 5: irreversible lanza incluso con require_confirmation=True
raised2 = False
try:
    registry.execute("send_refund", {"order_id": "ORD-005", "amount": 50.0}, require_confirmation=True)
except ConfirmationRequired:
    raised2 = True
assert raised2, "irreversible debe lanzar ConfirmationRequired incluso con require_confirmation=True"

# Test 6: herramienta no registrada lanza KeyError
raised3 = False
try:
    registry.execute("nonexistent_tool", {})
except KeyError:
    raised3 = True
assert raised3, "herramienta no registrada debe lanzar KeyError"

# Test 7: list_tools devuelve las 3 herramientas
tools = registry.list_tools()
assert len(tools) == 3, f"Debe haber 3 herramientas, hay {len(tools)}"
names = {t["name"] for t in tools}
assert names == {"get_order_status", "update_delivery_date", "send_refund"}

print("✅ Todos los tests del Ejercicio 1 pasaron")


---

## Ejercicio 2 — SelfCorrectingAgent: loop de autocorrección con límite de reintentos

**Lección de referencia:** 3.5 — Self-correction y reflexión

Un agente que no verifica su propio output entrega errores al sistema siguiente.
Un agente que reintenta indefinidamente puede quedar atascado para siempre.
El `SelfCorrectingAgent` combina un verifier con un límite de intentos:
reintenta pasando el feedback como contexto adicional, y escala si no converge.

### Lo que debes implementar

1. `SelfCorrectingAgent(llm_fn, verifier_fn, max_retries=3)` — recibe el LLM y el verifier como callables.
2. `execute(task)` — ejecuta el loop:
   - Llama a `llm_fn(task, feedback_history)` para obtener un output.
   - Llama a `verifier_fn(output)` → `(is_valid: bool, feedback: str)`.
   - Si válido: devuelve el output.
   - Si inválido: añade el feedback al historial y reintenta.
   - Tras `max_retries` fallidos: lanza `MaxRetriesExceeded`.

### Criterios de éxito
- El agente converge cuando el verifier acepta en el intento N ≤ max_retries.
- El feedback de cada intento fallido se acumula y se pasa al LLM en el siguiente intento.
- `MaxRetriesExceeded` incluye el número de intentos realizados.


In [ ]:
class MaxRetriesExceeded(Exception):
    """Raised when the agent exhausts all retry attempts without converging."""
    pass


class SelfCorrectingAgent:
    """
    Agente con loop de autocorrección basado en feedback del verifier.

    El verifier actúa como crítico externo. Si rechaza el output,
    el agente reintenta pasando el feedback como contexto adicional.
    El límite de reintentos evita loops infinitos.

    Lección 3.5: reflexión vale cuando el error requiere comprensión
    para ser detectado y no se corrige fácilmente en pasos posteriores.
    """

    def __init__(
        self,
        llm_fn: Callable[[str, List[str]], str],
        verifier_fn: Callable[[str], Tuple[bool, str]],
        max_retries: int = 3,
    ):
        """
        Args:
            llm_fn: callable(task, feedback_history) -> output_str
                    Simula la llamada al LLM. feedback_history es lista de strings
                    con el feedback de los intentos anteriores.
            verifier_fn: callable(output) -> (is_valid, feedback_str)
                    Verifica el output. Devuelve (True, "") si es válido.
            max_retries: número máximo de intentos antes de elevar MaxRetriesExceeded.
        """
        # TODO: almacena llm_fn, verifier_fn, max_retries
        pass

    def execute(self, task: str) -> str:
        """
        Ejecuta la tarea con el loop de autocorrección.

        Returns:
            El output validado por el verifier.

        Raises:
            MaxRetriesExceeded: si tras max_retries intentos el verifier
                                sigue rechazando el output.
        """
        feedback_history: List[str] = []
        # TODO: loop hasta max_retries
        #   1. output = self.llm_fn(task, feedback_history)
        #   2. is_valid, feedback = self.verifier_fn(output)
        #   3. si is_valid: return output
        #   4. si no: feedback_history.append(feedback) y continúa
        # TODO: si agota reintentos: raise MaxRetriesExceeded con el número de intentos
        pass


In [ ]:
# ── Tests Ejercicio 2 ────────────────────────────────────────────────────────

# Scenario A: verifier falla 2 veces, acepta en el 3er intento → debe converger
call_count_a = 0

def llm_fn_a(task: str, feedback_history: List[str]) -> str:
    global call_count_a
    call_count_a += 1
    if call_count_a < 3:
        return f"bad_output_{call_count_a}"  # salida mala
    return "good_output"  # 3er intento: correcto

def verifier_fn_a(output: str) -> Tuple[bool, str]:
    if output == "good_output":
        return True, ""
    return False, f"Output '{output}' no cumple criterios. Debe ser 'good_output'."

call_count_a = 0
agent_a = SelfCorrectingAgent(llm_fn_a, verifier_fn_a, max_retries=3)
result_a = agent_a.execute("genera output correcto")
assert result_a == "good_output", f"Debería haber convergido en 'good_output', obtuvo '{result_a}'"
assert call_count_a == 3, f"Debería haber llamado al LLM 3 veces, llamó {call_count_a}"

# Test: el feedback se acumula y se pasa al LLM
received_feedback = []
call_count_b = 0

def llm_fn_b(task: str, feedback_history: List[str]) -> str:
    global call_count_b
    call_count_b += 1
    received_feedback.append(list(feedback_history))  # captura copia
    return "good" if call_count_b >= 2 else "bad"

def verifier_fn_b(output: str) -> Tuple[bool, str]:
    return (True, "") if output == "good" else (False, "necesita corrección")

call_count_b = 0
agent_b = SelfCorrectingAgent(llm_fn_b, verifier_fn_b, max_retries=3)
agent_b.execute("tarea")
# En el 1er intento, el feedback_history debe estar vacío
assert received_feedback[0] == [], "Primer intento debe tener feedback_history vacío"
# En el 2do intento, debe contener el feedback del 1er fallo
assert len(received_feedback[1]) == 1, "Segundo intento debe tener 1 feedback acumulado"
assert "necesita corrección" in received_feedback[1][0]

# Scenario B: verifier siempre falla → debe lanzar MaxRetriesExceeded tras 3 intentos
call_count_c = 0

def llm_fn_c(task: str, feedback_history: List[str]) -> str:
    global call_count_c
    call_count_c += 1
    return "always_bad"

def verifier_fn_c(output: str) -> Tuple[bool, str]:
    return False, "siempre falla"

call_count_c = 0
agent_c = SelfCorrectingAgent(llm_fn_c, verifier_fn_c, max_retries=3)
raised = False
try:
    agent_c.execute("tarea imposible")
except MaxRetriesExceeded as e:
    raised = True
    assert "3" in str(e), f"MaxRetriesExceeded debe mencionar el número de intentos, fue: {e}"
assert raised, "Debe lanzar MaxRetriesExceeded cuando el verifier siempre falla"
assert call_count_c == 3, f"Debe llamar al LLM exactamente max_retries=3 veces, llamó {call_count_c}"

print("✅ Todos los tests del Ejercicio 2 pasaron")


---

## Ejercicio 3 — AgentSupervisor: supervisión con escalado a humano

**Lección de referencia:** 3.4 — Supervisors y supervisión humana

Un supervisor bien diseñado no escala todo (sobrecarga al humano) ni nada (autonomía ciega).
Observa el estado del agente en cada paso y toma una de tres decisiones:
continuar, forzar retry, o escalar a supervisión humana.
Los criterios son objetivos y predecibles, no heurísticos.

### Lo que debes implementar

`AgentSupervisor.observe(agent_state)` recibe un dict con:
- `steps_taken` (int): pasos ejecutados hasta ahora.
- `time_elapsed` (float): segundos transcurridos.
- `last_outputs` (List[str]): últimos outputs del agente (orden cronológico).

Devuelve: `"continue"`, `"retry"`, o `"escalate_to_human"`.

**Reglas (por prioridad, de mayor a menor):**
1. `steps_taken > MAX_STEPS` (20) → `"escalate_to_human"`
2. `time_elapsed > MAX_TIME_SECONDS` (30) → `"escalate_to_human"`
3. Las últimas 3 outputs **todas** contienen la palabra `"error"` → `"retry"`
4. En cualquier otro caso → `"continue"`

### Criterios de éxito
- Cada regla se dispara exactamente cuando se cumple su condición.
- Las reglas tienen la precedencia correcta (steps y time van antes que retry).


In [ ]:
@dataclass
class AgentState:
    steps_taken: int
    time_elapsed: float
    last_outputs: List[str] = field(default_factory=list)


class AgentSupervisor:
    """
    Supervisor que observa el estado del agente y decide cuándo intervenir.

    Implementa supervisión selectiva: solo escala los casos que realmente
    necesitan atención humana, evitando el teatro de supervisión donde
    todo escala y nada se revisa realmente (lección 3.4).
    """

    MAX_STEPS = 20
    MAX_TIME_SECONDS = 30
    CONSECUTIVE_ERRORS_THRESHOLD = 3

    def observe(self, agent_state: AgentState) -> str:
        """
        Observa el estado actual del agente y devuelve una decisión.

        Returns:
            "escalate_to_human": el agente necesita supervisión humana inmediata.
            "retry": el agente debe reiniciar el paso actual con contexto limpio.
            "continue": el agente puede continuar de forma autónoma.
        """
        # TODO: regla 1 — steps excedidos → escalate
        # TODO: regla 2 — tiempo excedido → escalate
        # TODO: regla 3 — últimas CONSECUTIVE_ERRORS_THRESHOLD outputs contienen "error" → retry
        #   Pista: comprueba que len(last_outputs) >= threshold
        #          y que todas las últimas N contienen "error" (case-insensitive)
        # TODO: default → continue
        pass


In [ ]:
# ── Tests Ejercicio 3 ────────────────────────────────────────────────────────

supervisor = AgentSupervisor()

# Test 1: caso normal → continue
state_ok = AgentState(steps_taken=5, time_elapsed=10.0, last_outputs=["ok", "done", "success"])
assert supervisor.observe(state_ok) == "continue", "Estado normal debe devolver 'continue'"

# Test 2: steps excedidos → escalate
state_steps = AgentState(steps_taken=21, time_elapsed=10.0, last_outputs=["ok"])
assert supervisor.observe(state_steps) == "escalate_to_human", "steps > MAX_STEPS debe escalar"

# Test 3: exactamente en el límite de steps (20) → continue (el límite es MAYOR QUE)
state_steps_limit = AgentState(steps_taken=20, time_elapsed=10.0, last_outputs=["ok"])
assert supervisor.observe(state_steps_limit) == "continue", "steps == MAX_STEPS no debe escalar (es >)"

# Test 4: tiempo excedido → escalate
state_time = AgentState(steps_taken=5, time_elapsed=31.0, last_outputs=["ok"])
assert supervisor.observe(state_time) == "escalate_to_human", "time > MAX_TIME_SECONDS debe escalar"

# Test 5: steps excedidos tiene precedencia sobre errores consecutivos
state_both = AgentState(steps_taken=25, time_elapsed=5.0, last_outputs=["error1", "error2", "error3"])
assert supervisor.observe(state_both) == "escalate_to_human", "steps excedidos debe ganar sobre errores"

# Test 6: 3 últimos outputs con 'error' → retry
state_errors = AgentState(
    steps_taken=5, time_elapsed=10.0,
    last_outputs=["success", "error: timeout", "error: invalid", "error: retry"]
)
assert supervisor.observe(state_errors) == "retry", "3 últimos con 'error' debe devolver 'retry'"

# Test 7: solo 2 últimos con 'error' → continue (no llega al threshold)
state_two_errors = AgentState(
    steps_taken=5, time_elapsed=10.0,
    last_outputs=["success", "success", "error: x", "error: y"]
)
# Solo hay 2 errors al final, no 3 → continue
# Nota: depende de la implementación exacta. Con threshold=3 y solo 2 al final → continue
result_two = supervisor.observe(state_two_errors)
assert result_two == "continue", f"Solo 2 últimos errores no debe disparar retry, obtuvo '{result_two}'"

# Test 8: last_outputs vacío → continue sin crash
state_empty = AgentState(steps_taken=1, time_elapsed=0.5, last_outputs=[])
assert supervisor.observe(state_empty) == "continue", "last_outputs vacío debe devolver 'continue'"

# Test 9: 'error' en minúsculas y mayúsculas → case-insensitive
state_case = AgentState(
    steps_taken=5, time_elapsed=10.0,
    last_outputs=["ERROR: fatal", "Error: timeout", "error en proceso"]
)
assert supervisor.observe(state_case) == "retry", "La detección de 'error' debe ser case-insensitive"

print("✅ Todos los tests del Ejercicio 3 pasaron")


---

## Ejercicio 4 — AgentMemory: memoria por capas con ciclo de vida correcto

**Lección de referencia:** 3.6 — Memoria en agentes: short vs long term

Un agente sin memoria empieza desde cero en cada sesión.
Un agente que mezcla tipos de memoria acaba con un almacén que crece sin control.
La clave es definir el ciclo de vida de cada tipo antes de elegir el almacén.

### Lo que debes implementar

`AgentMemory(storage_path)` con tres capas independientes:

| Capa | Persistencia | Almacén | Ciclo de vida |
|---|---|---|---|
| `working_memory` | Solo en RAM | `dict` | Se pierde al destruir la instancia |
| `episodic_memory` | Disco (JSON) | `list` de dicts | Persiste entre instancias |
| `semantic_memory` | Disco (JSON) | `dict` de facts | Persiste entre instancias |

**Métodos requeridos:**
- `store_working(key, value)` / `get_working(key, default=None)`
- `store_episode(event_dict)` / `get_recent_episodes(n=5)`
- `store_fact(key, value)` / `get_fact(key, default=None)`

### Criterios de éxito
- `working_memory` NO persiste entre instancias de `AgentMemory`.
- `episodic_memory` y `semantic_memory` SÍ persisten entre instancias.
- `get_recent_episodes(n)` devuelve los últimos `n` episodios (más reciente al final o al principio, define tú el orden pero sé consistente).


In [ ]:
class AgentMemory:
    """
    Capa de memoria de un agente con tres tipos diferenciados por ciclo de vida.

    Lección 3.6: define el ciclo de vida antes de elegir el almacén.
    - working: atado a la sesión (RAM)
    - episodic: atado al usuario/contexto (disco, persiste entre sesiones)
    - semantic: conocimiento del dominio (disco, raramente cambia)
    """

    EPISODIC_FILE = "episodic_memory.json"
    SEMANTIC_FILE = "semantic_memory.json"

    def __init__(self, storage_path: str = "."):
        """
        Args:
            storage_path: directorio donde se persisten episodic y semantic.
        """
        self._storage = Path(storage_path)
        self._storage.mkdir(parents=True, exist_ok=True)

        # TODO: inicializa working_memory como dict vacío en RAM
        # TODO: carga episodic_memory desde disco (lista vacía si no existe el archivo)
        # TODO: carga semantic_memory desde disco (dict vacío si no existe el archivo)
        pass

    # ── Working memory (solo RAM) ──────────────────────────────────────────

    def store_working(self, key: str, value: Any) -> None:
        """Almacena un valor en working memory (solo dura en esta instancia)."""
        # TODO
        pass

    def get_working(self, key: str, default: Any = None) -> Any:
        """Recupera un valor de working memory."""
        # TODO
        pass

    # ── Episodic memory (persiste en disco) ───────────────────────────────

    def store_episode(self, event_dict: Dict) -> None:
        """
        Añade un episodio a la memoria episódica y persiste en disco.

        Args:
            event_dict: dict con al menos 'timestamp' y 'description'.
                        Puede tener cualquier clave adicional.
        """
        # TODO: añade event_dict a self._episodes
        # TODO: persiste la lista completa en EPISODIC_FILE
        pass

    def get_recent_episodes(self, n: int = 5) -> List[Dict]:
        """
        Devuelve los últimos n episodios (más reciente último).

        Returns:
            Lista de hasta n episodios en orden cronológico (más antiguo primero).
        """
        # TODO: devuelve los últimos n elementos de self._episodes
        pass

    # ── Semantic memory (persiste en disco) ───────────────────────────────

    def store_fact(self, key: str, value: Any) -> None:
        """
        Almacena un hecho en memoria semántica y persiste en disco.

        Semantic memory es el conocimiento del dominio: políticas, procedimientos,
        configuraciones. El agente solo lee, raramente escribe.
        """
        # TODO: almacena en self._facts y persiste en SEMANTIC_FILE
        pass

    def get_fact(self, key: str, default: Any = None) -> Any:
        """Recupera un hecho de memoria semántica."""
        # TODO
        pass

    # ── Helpers internos ──────────────────────────────────────────────────

    def _save_json(self, filename: str, data: Any) -> None:
        """Persiste data como JSON en storage_path/filename."""
        with open(self._storage / filename, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    def _load_json(self, filename: str, default: Any) -> Any:
        """Carga JSON desde storage_path/filename, devuelve default si no existe."""
        path = self._storage / filename
        if not path.exists():
            return default
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)


In [ ]:
# ── Tests Ejercicio 4 ────────────────────────────────────────────────────────

# Usamos un directorio temporal para no contaminar el entorno
tmp_dir = tempfile.mkdtemp(prefix="agent_memory_test_")

try:
    # ── Working memory ────────────────────────────────────────────────────
    mem1 = AgentMemory(tmp_dir)
    mem1.store_working("current_task", "analyze_report")
    mem1.store_working("step", 3)

    assert mem1.get_working("current_task") == "analyze_report"
    assert mem1.get_working("step") == 3
    assert mem1.get_working("nonexistent") is None, "default debe ser None"
    assert mem1.get_working("nonexistent", "fallback") == "fallback", "default personalizado debe funcionar"

    # Working memory NO persiste entre instancias
    mem2 = AgentMemory(tmp_dir)  # nueva instancia, misma ruta
    assert mem2.get_working("current_task") is None, \
        "working_memory NO debe persistir entre instancias (vive solo en RAM)"

    # ── Episodic memory ───────────────────────────────────────────────────
    mem1.store_episode({"timestamp": "2025-01-01T10:00", "description": "Consulta inicial", "user": "alice"})
    mem1.store_episode({"timestamp": "2025-01-01T10:05", "description": "Error al cargar datos", "user": "alice"})
    mem1.store_episode({"timestamp": "2025-01-01T10:10", "description": "Tarea completada", "user": "alice"})

    episodes = mem1.get_recent_episodes(2)
    assert len(episodes) == 2, f"get_recent_episodes(2) debe devolver 2, devolvió {len(episodes)}"
    # El más reciente debe estar en la lista
    descriptions = [e["description"] for e in episodes]
    assert "Tarea completada" in descriptions, "El episodio más reciente debe estar en get_recent_episodes(2)"

    # Episodic persiste entre instancias
    mem3 = AgentMemory(tmp_dir)
    episodes_reloaded = mem3.get_recent_episodes(10)
    assert len(episodes_reloaded) == 3, \
        f"Episodic memory debe persistir: esperaba 3 episodios, encontró {len(episodes_reloaded)}"

    # ── Semantic memory ───────────────────────────────────────────────────
    mem1.store_fact("escalation_policy", "Escalar a L2 si el tiempo supera 30 minutos")
    mem1.store_fact("max_retries", 3)

    assert mem1.get_fact("escalation_policy") == "Escalar a L2 si el tiempo supera 30 minutos"
    assert mem1.get_fact("max_retries") == 3
    assert mem1.get_fact("unknown_fact") is None
    assert mem1.get_fact("unknown_fact", "default_policy") == "default_policy"

    # Semantic persiste entre instancias
    mem4 = AgentMemory(tmp_dir)
    assert mem4.get_fact("escalation_policy") == "Escalar a L2 si el tiempo supera 30 minutos", \
        "semantic_memory debe persistir entre instancias"
    assert mem4.get_fact("max_retries") == 3

    # ── get_recent_episodes con n > total ────────────────────────────────
    all_episodes = mem1.get_recent_episodes(100)
    assert len(all_episodes) == 3, \
        f"get_recent_episodes(100) con solo 3 episodios debe devolver 3, devolvió {len(all_episodes)}"

    print("✅ Todos los tests del Ejercicio 4 pasaron")

finally:
    # Limpia el directorio temporal
    shutil.rmtree(tmp_dir, ignore_errors=True)


---

## Tips y referencias

### Patrones clave de S3

**Ejercicio 1 — ToolRegistry:**
- El principio de mínimo privilegio no es solo seguridad: reduce la superficie de error del LLM al tomar decisiones.
- En producción, `ConfirmationRequired` debería incluir suficiente contexto para que el supervisor humano decida sin ver el código.
- Extiende con `dry_run=True` para simular acciones irreversibles sin ejecutarlas.

**Ejercicio 2 — SelfCorrectingAgent:**
- El prompt del crítico (`verifier_fn`) debe ser **distinto** al del generador (`llm_fn`). Mismo modelo, distinto rol = distinto comportamiento.
- `feedback_history` acumulado previene que el agente repita el mismo error en cada intento.
- En producción: añade `delay` entre reintentos para evitar rate-limiting y dar tiempo al sistema a recuperarse.

**Ejercicio 3 — AgentSupervisor:**
- La supervisión bien diseñada hace que las aprobaciones humanas sean **raras y significativas**.
- `observe()` puede extenderse con una función de scoring que combine varias señales débiles.
- Loguea cada decisión del supervisor: es tu principal herramienta de debugging en producción.

**Ejercicio 4 — AgentMemory:**
- Define el ciclo de vida **antes** de elegir el almacén. La tecnología es secundaria.
- `episodic_memory` debe tener un límite máximo de entradas para evitar que crezca sin control (FIFO o resumen).
- `semantic_memory` solo debería ser escrita por procesos de administración, no por el agente en runtime.

### Lecciones del módulo

| Lección | Concepto central |
|---|---|
| 3.1 Qué es un agente | El loop: percibe → razona → actúa → observa |
| 3.2 Skills y herramientas | Principio de mínimo privilegio en el espacio de acción |
| 3.3 Verifiers | Verifier determinista para errores de estructura; LLM-critic para razonamiento |
| 3.4 Supervisores | human-in-the-loop para irreversible; human-on-the-loop para reversible de coste medio |
| 3.5 Self-correction | Prompt del crítico distinto al del generador; límite de iteraciones siempre presente |
| 3.6 Memoria | Ciclo de vida: sesión → working; usuario → episodic; dominio → semantic |
| 3.7 Comunicación | Patrones: orquestador-trabajador, peer-to-peer, blackboard |
